# Top-5 Model Ensemble

Loads selected trained checkpoints and evaluates majority voting, soft voting, and weighted soft voting on the IDRiD test set. No model is trained in this notebook.

## 1. Setup

In [ ]:
import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

try:
    import cv2
except ImportError:
    cv2 = None

In [ ]:
# ============================================================
# Configuration
# Attach the Kaggle outputs/datasets containing the .pth checkpoint files.
# The notebook searches /kaggle/input and /kaggle/working by filename.
# ============================================================

DATA_ROOT = Path("/kaggle/input/datasets/paulconrad21/eye-disease-dataset/eye-disease-data")

TRAIN_CSV_PATH = DATA_ROOT / "2. Groundtruths" / "a. IDRiD_Disease Grading_Training Labels.csv"
TEST_CSV_PATH  = DATA_ROOT / "2. Groundtruths" / "b. IDRiD_Disease Grading_Testing Labels.csv"
TRAIN_IMAGE_DIR = DATA_ROOT / "1. Original Images" / "a. Training Set"
TEST_IMAGE_DIR  = DATA_ROOT / "1. Original Images" / "b. Testing Set"

IMAGE_COL = "Image name"
LABEL_COL = "Retinopathy grade"
NUM_CLASSES = 5
IMAGE_SIZE = 384
BATCH_SIZE = 8
NUM_WORKERS = 2
SEED = 42
VAL_SIZE = 0.2
USE_STRATIFIED_SPLIT = True
DEVICE_MODE = "cuda"  # "cuda", "auto", or "cpu"

ENSEMBLE_SUMMARY_CSV = "/kaggle/working/top5_ensemble_summary.csv"
ENSEMBLE_PREDICTIONS_CSV = "/kaggle/working/top5_ensemble_predictions.csv"
INDIVIDUAL_MODEL_CSV = "/kaggle/working/top5_individual_model_metrics.csv"

# Selected from prior experiment results:
# - B2-4: best test Macro F1 / accuracy
# - B2-5: best test Kappa
# - B2-6: weighted CE / imbalance-focused B2
# - R7: best ResNet, different architecture
# - B0-7: best EfficientNet-B0 fine-tuning run, different capacity
MODEL_SPECS = [
    {
        "run_id": "B2-4",
        "model": "efficientnet_b2",
        "checkpoint_filename": "b2_4_best_efficientnet_b2.pth",
        "preprocessing_preset": "none",
        "use_custom_head": True,
        "hidden_dim": 256,
        "dropout": 0.20,
        "soft_vote_weight": 1.30,
    },
    {
        "run_id": "B2-5",
        "model": "efficientnet_b2",
        "checkpoint_filename": "b2_5_best_efficientnet_b2.pth",
        "preprocessing_preset": "none",
        "use_custom_head": True,
        "hidden_dim": 256,
        "dropout": 0.20,
        "soft_vote_weight": 1.20,
    },
    {
        "run_id": "B2-6",
        "model": "efficientnet_b2",
        "checkpoint_filename": "b2_6_best_efficientnet_b2.pth",
        "preprocessing_preset": "none",
        "use_custom_head": True,
        "hidden_dim": 256,
        "dropout": 0.20,
        "soft_vote_weight": 1.10,
    },
    {
        "run_id": "R7",
        "model": "resnet18",
        "checkpoint_filename": "r7_best_resnet18.pth",
        "preprocessing_preset": "crop_graham",
        "use_custom_head": True,
        "hidden_dim": 512,
        "dropout": 0.20,
        "soft_vote_weight": 1.00,
    },
    {
        "run_id": "B0-7",
        "model": "efficientnet_b0",
        "checkpoint_filename": "b0_7_best_efficientnet_b0.pth",
        "preprocessing_preset": "crop_graham",
        "use_custom_head": True,
        "hidden_dim": 256,
        "dropout": 0.20,
        "soft_vote_weight": 1.00,
    },
]


def get_device():
    if DEVICE_MODE == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if DEVICE_MODE == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable. Set DEVICE_MODE='cpu' if needed.")
        return torch.device("cuda")
    if DEVICE_MODE == "cpu":
        return torch.device("cpu")
    raise ValueError(f"Invalid DEVICE_MODE: {DEVICE_MODE}")

DEVICE = get_device()
print("Device:", DEVICE)

## 2. Data And Preprocessing

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

train_full_df = pd.read_csv(TRAIN_CSV_PATH)[[IMAGE_COL, LABEL_COL]].copy()
test_df = pd.read_csv(TEST_CSV_PATH)[[IMAGE_COL, LABEL_COL]].copy()
train_full_df[LABEL_COL] = train_full_df[LABEL_COL].astype(int)
test_df[LABEL_COL] = test_df[LABEL_COL].astype(int)

if USE_STRATIFIED_SPLIT:
    _, val_df = train_test_split(
        train_full_df,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=train_full_df[LABEL_COL],
    )
else:
    _, val_df = train_test_split(
        train_full_df,
        test_size=VAL_SIZE,
        random_state=SEED,
    )

print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))
print("Test distribution:")
print(test_df[LABEL_COL].value_counts().sort_index())

In [ ]:
def crop_black_background(image, threshold=10, padding=16):
    gray = np.array(image.convert("L"))
    mask = gray > threshold
    if not mask.any():
        return image

    ys, xs = np.where(mask)
    left = max(int(xs.min()) - padding, 0)
    right = min(int(xs.max()) + padding + 1, image.width)
    top = max(int(ys.min()) - padding, 0)
    bottom = min(int(ys.max()) + padding + 1, image.height)

    if right <= left or bottom <= top:
        return image

    return image.crop((left, top, right, bottom))


def apply_graham_preprocessing(image):
    if cv2 is None:
        raise ImportError("OpenCV/cv2 is required for crop_graham preprocessing.")

    arr = np.array(image)
    blurred = cv2.GaussianBlur(arr, (0, 0), 10)
    enhanced = cv2.addWeighted(arr, 4.0, blurred, -4.0, 128)
    return Image.fromarray(np.clip(enhanced, 0, 255).astype(np.uint8))


def preprocess_image(image, preset):
    if preset == "none":
        return image
    if preset == "crop_graham":
        image = crop_black_background(image)
        image = apply_graham_preprocessing(image)
        return image
    raise ValueError(f"Unsupported preprocessing_preset for ensemble: {preset}")


class FundusEvalDataset(Dataset):
    def __init__(self, dataframe, image_dir, preprocessing_preset="none"):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.preprocessing_preset = preprocessing_preset
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_name = str(row[IMAGE_COL])
        if not image_name.lower().endswith((".jpg", ".jpeg", ".png")):
            image_name += ".jpg"

        image_path = self.image_dir / image_name
        image = Image.open(image_path).convert("RGB")
        image = preprocess_image(image, self.preprocessing_preset)
        image = self.transform(image)
        label = int(row[LABEL_COL])
        return image, label, image_name


def make_loader(dataframe, image_dir, preprocessing_preset):
    dataset = FundusEvalDataset(
        dataframe=dataframe,
        image_dir=image_dir,
        preprocessing_preset=preprocessing_preset,
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
    )

## 3. Model Loading

In [ ]:
def create_classification_head(in_features, num_classes, dropout, use_custom_head, hidden_dim, activation="silu"):
    if use_custom_head:
        act = nn.SiLU(inplace=True) if activation == "silu" else nn.ReLU(inplace=True)
        return nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            act,
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    if dropout > 0:
        return nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))

    return nn.Linear(in_features, num_classes)


def create_model_from_spec(spec):
    model_name = spec["model"]
    dropout = spec["dropout"]
    hidden_dim = spec["hidden_dim"]
    use_custom_head = spec["use_custom_head"]

    if model_name == "resnet18":
        model = models.resnet18(weights=None)
        in_features = model.fc.in_features
        model.fc = create_classification_head(
            in_features,
            NUM_CLASSES,
            dropout=dropout,
            use_custom_head=use_custom_head,
            hidden_dim=hidden_dim,
            activation="relu",
        )
        return model

    if model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        in_features = model.classifier[-1].in_features
        model.classifier = create_classification_head(
            in_features,
            NUM_CLASSES,
            dropout=dropout,
            use_custom_head=use_custom_head,
            hidden_dim=hidden_dim,
            activation="silu",
        )
        return model

    if model_name == "efficientnet_b2":
        model = models.efficientnet_b2(weights=None)
        in_features = model.classifier[-1].in_features
        model.classifier = create_classification_head(
            in_features,
            NUM_CLASSES,
            dropout=dropout,
            use_custom_head=use_custom_head,
            hidden_dim=hidden_dim,
            activation="silu",
        )
        return model

    raise ValueError(f"Unsupported model: {model_name}")


def find_checkpoint(filename):
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    matches = []
    for root in search_roots:
        if root.exists():
            matches.extend(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Checkpoint not found: {filename}\n"
            "Attach the Kaggle output dataset containing this .pth file, "
            "or copy it into /kaggle/working."
        )

    # Prefer /kaggle/input for saved outputs, but any exact filename is fine.
    return sorted(matches, key=lambda p: str(p))[0]


def load_checkpoint_state(path):
    checkpoint = torch.load(path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        return checkpoint["state_dict"]
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]
    return checkpoint


def load_model(spec):
    checkpoint_path = find_checkpoint(spec["checkpoint_filename"])
    model = create_model_from_spec(spec)
    state_dict = load_checkpoint_state(checkpoint_path)
    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()
    print(f"Loaded {spec['run_id']} from {checkpoint_path}")
    return model, checkpoint_path

## 4. Prediction Helpers

In [ ]:
@torch.no_grad()
def predict_model(model, loader):
    all_probs = []
    all_preds = []
    all_labels = []
    all_names = []

    for images, labels, names in tqdm(loader, desc="Predicting", leave=False):
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.extend(labels.numpy().tolist())
        all_names.extend(list(names))

    probs = np.concatenate(all_probs, axis=0)
    preds = np.concatenate(all_preds, axis=0)
    labels = np.array(all_labels)
    return probs, preds, labels, all_names


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
    }


def majority_vote(pred_matrix, tie_break_probs=None):
    final_preds = []
    for i in range(pred_matrix.shape[0]):
        votes = pred_matrix[i].tolist()
        counts = Counter(votes)
        max_count = max(counts.values())
        winners = [cls for cls, count in counts.items() if count == max_count]

        if len(winners) == 1:
            final_preds.append(winners[0])
        elif tie_break_probs is not None:
            winner_scores = {cls: tie_break_probs[i, cls] for cls in winners}
            final_preds.append(max(winner_scores, key=winner_scores.get))
        else:
            final_preds.append(winners[0])

    return np.array(final_preds)


def evaluate_predictions(name, y_true, y_pred):
    metrics = compute_metrics(y_true, y_pred)
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}")
    print(f"Kappa:    {metrics['kappa']:.4f}")
    print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)), digits=4, zero_division=0))
    return metrics


def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=list(range(NUM_CLASSES)), yticklabels=list(range(NUM_CLASSES)))
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

## 5. Run Ensemble Evaluation

In [ ]:
def run_ensemble_for_split(split_name, dataframe, image_dir):
    model_probs = []
    model_preds = []
    model_names = []
    individual_rows = []
    labels_reference = None
    image_names_reference = None

    for spec in MODEL_SPECS:
        model, checkpoint_path = load_model(spec)
        loader = make_loader(dataframe, image_dir, spec["preprocessing_preset"])
        probs, preds, labels, image_names = predict_model(model, loader)

        if labels_reference is None:
            labels_reference = labels
            image_names_reference = image_names
        else:
            assert np.array_equal(labels_reference, labels), "Label order mismatch."
            assert image_names_reference == image_names, "Image order mismatch."

        metrics = compute_metrics(labels, preds)
        individual_rows.append({
            "split": split_name,
            "run_id": spec["run_id"],
            "model": spec["model"],
            "checkpoint_filename": spec["checkpoint_filename"],
            "checkpoint_path": str(checkpoint_path),
            "preprocessing_preset": spec["preprocessing_preset"],
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
            "kappa": metrics["kappa"],
            "soft_vote_weight": spec["soft_vote_weight"],
        })

        model_probs.append(probs)
        model_preds.append(preds)
        model_names.append(spec["run_id"])

        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    probs_stack = np.stack(model_probs, axis=0)  # [num_models, num_samples, num_classes]
    preds_stack = np.stack(model_preds, axis=1)  # [num_samples, num_models]
    weights = np.array([spec["soft_vote_weight"] for spec in MODEL_SPECS], dtype=np.float32)

    soft_probs = probs_stack.mean(axis=0)
    weighted_soft_probs = np.average(probs_stack, axis=0, weights=weights)

    soft_preds = np.argmax(soft_probs, axis=1)
    weighted_soft_preds = np.argmax(weighted_soft_probs, axis=1)
    majority_preds = majority_vote(preds_stack, tie_break_probs=weighted_soft_probs)

    ensemble_outputs = {
        "majority_vote": majority_preds,
        "soft_vote": soft_preds,
        "weighted_soft_vote": weighted_soft_preds,
    }

    summary_rows = []
    for method_name, ensemble_preds in ensemble_outputs.items():
        metrics = evaluate_predictions(f"{split_name}: {method_name}", labels_reference, ensemble_preds)
        plot_confusion(labels_reference, ensemble_preds, f"{split_name}: {method_name}")
        summary_rows.append({
            "split": split_name,
            "method": method_name,
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
            "kappa": metrics["kappa"],
            "models": ",".join(model_names),
        })

    predictions_df = pd.DataFrame({
        "split": split_name,
        "image_name": image_names_reference,
        "true_label": labels_reference,
        "majority_vote_pred": majority_preds,
        "soft_vote_pred": soft_preds,
        "weighted_soft_vote_pred": weighted_soft_preds,
    })

    for model_index, model_name in enumerate(model_names):
        predictions_df[f"{model_name}_pred"] = preds_stack[:, model_index]
        for class_index in range(NUM_CLASSES):
            predictions_df[f"{model_name}_prob_{class_index}"] = probs_stack[model_index, :, class_index]

    return pd.DataFrame(summary_rows), pd.DataFrame(individual_rows), predictions_df


val_summary, val_individual, val_predictions = run_ensemble_for_split("validation", val_df, TRAIN_IMAGE_DIR)
test_summary, test_individual, test_predictions = run_ensemble_for_split("test", test_df, TEST_IMAGE_DIR)

summary_df = pd.concat([val_summary, test_summary], ignore_index=True)
individual_df = pd.concat([val_individual, test_individual], ignore_index=True)
predictions_df = pd.concat([val_predictions, test_predictions], ignore_index=True)

summary_df.to_csv(ENSEMBLE_SUMMARY_CSV, index=False)
individual_df.to_csv(INDIVIDUAL_MODEL_CSV, index=False)
predictions_df.to_csv(ENSEMBLE_PREDICTIONS_CSV, index=False)

print("Saved ensemble summary to:", ENSEMBLE_SUMMARY_CSV)
print("Saved individual model metrics to:", INDIVIDUAL_MODEL_CSV)
print("Saved ensemble predictions to:", ENSEMBLE_PREDICTIONS_CSV)

display(summary_df)
display(individual_df)